In [1]:
#Step 1: Load Dataset
import pandas as pd

df = pd.read_csv("Groceries_dataset.csv")
print(df.head())

   Member_number        Date   itemDescription
0           1808  21-07-2015    tropical fruit
1           2552  05-01-2015        whole milk
2           2300  19-09-2015         pip fruit
3           1187  12-12-2015  other vegetables
4           3037  01-02-2015        whole milk


In [2]:
#Step 2: Convert into Transaction Format
transactions = df.groupby(['Member_number', 'Date'])['itemDescription'].apply(list).values.tolist()

print(transactions[:5])

[['sausage', 'whole milk', 'semi-finished bread', 'yogurt'], ['whole milk', 'pastry', 'salty snack'], ['canned beer', 'misc. beverages'], ['sausage', 'hygiene articles'], ['soda', 'pickled vegetables']]


In [4]:
#Step 3: Apriori From Scratch
from itertools import combinations

def get_support(itemset, transactions):
    count = 0
    for transaction in transactions:
        if set(itemset).issubset(set(transaction)):
            count += 1
    return count / len(transactions)


def apriori(transactions, min_support):
    items = set(item for transaction in transactions for item in transaction)
    frequent_itemsets = {}
    
    # 1-itemsets
    L1 = {}
    for item in items:
        support = get_support([item], transactions)
        if support >= min_support:
            L1[frozenset([item])] = support
    
    k = 2
    current_L = L1
    
    while current_L:
        frequent_itemsets.update(current_L)
        candidates = set()
        itemsets = list(current_L.keys())
        
        for i in range(len(itemsets)):
            for j in range(i+1, len(itemsets)):
                union = itemsets[i].union(itemsets[j])
                if len(union) == k:
                    candidates.add(union)
        
        next_L = {}
        for candidate in candidates:
            support = get_support(candidate, transactions)
            if support >= min_support:
                next_L[candidate] = support
        
        current_L = next_L
        k += 1
    
    return frequent_itemsets

In [5]:
#Run Apriori
frequent_itemsets = apriori(transactions, min_support=0.02)

for itemset, support in list(frequent_itemsets.items())[:10]:
    print(itemset, support)

frozenset({'frankfurter'}) 0.037759807525228894
frozenset({'pork'}) 0.037091492347791216
frozenset({'soda'}) 0.09710619528169484
frozenset({'white bread'}) 0.023992514870012697
frozenset({'chocolate'}) 0.02359152576355009
frozenset({'rolls/buns'}) 0.11000467820624206
frozenset({'whipped/sour cream'}) 0.043707812604424245
frozenset({'yogurt'}) 0.08587850030074183
frozenset({'napkins'}) 0.022121232373187194
frozenset({'citrus fruit'}) 0.05313105660629553


In [9]:
#Generate Association Rules (From Scratch)
def generate_rules(frequent_itemsets, min_confidence):
    rules = []
    
    for itemset in frequent_itemsets:
        if len(itemset) > 1:
            for item in itemset:
                antecedent = frozenset([item])
                consequent = itemset - antecedent
                
                support_itemset = frequent_itemsets[itemset]
                support_antecedent = frequent_itemsets[antecedent]
                
                confidence = support_itemset / support_antecedent
                
                if confidence >= min_confidence:
                    rules.append((antecedent, consequent, confidence))
    
    return rules

In [10]:
rules = generate_rules(frequent_itemsets, min_confidence=0.3)

print(rules[:10])

[]


In [12]:
rules = generate_rules(frequent_itemsets, min_confidence=0.1)

print(len(rules))
print(rules[:5])

0
[]


In [14]:
print(len(frequent_itemsets))

38


In [15]:
frequent_itemsets = apriori(transactions, min_support=0.01)
print(len(frequent_itemsets))

69


In [16]:
print("Frequent Itemsets:", len(frequent_itemsets))

rules = generate_rules(frequent_itemsets, min_confidence=0.3)

print("Generated Rules:", len(rules))
print(rules[:10])

Frequent Itemsets: 69
Generated Rules: 0
[]


In [17]:
print(len(frequent_itemsets))
print(len(rules))

69
0


In [18]:
rules = generate_rules(frequent_itemsets, min_confidence=0.1)

print(len(rules))
print(rules[:10])

4
[(frozenset({'other vegetables'}), frozenset({'whole milk'}), 0.12151067323481116), (frozenset({'yogurt'}), frozenset({'whole milk'}), 0.1299610894941634), (frozenset({'rolls/buns'}), frozenset({'whole milk'}), 0.12697448359659783), (frozenset({'soda'}), frozenset({'whole milk'}), 0.11975223675154853)]


In [21]:
for antecedent, consequent, confidence in rules:
    print(f"{set(antecedent)} → {set(consequent)} | Confidence: {confidence:.2f}")

{'other vegetables'} → {'whole milk'} | Confidence: 0.12
{'yogurt'} → {'whole milk'} | Confidence: 0.13
{'rolls/buns'} → {'whole milk'} | Confidence: 0.13
{'soda'} → {'whole milk'} | Confidence: 0.12


In [22]:
def generate_rules_with_lift(frequent_itemsets, transactions, min_confidence):
    rules = []
    total_transactions = len(transactions)

    for itemset in frequent_itemsets:
        if len(itemset) > 1:
            for item in itemset:
                antecedent = frozenset([item])
                consequent = itemset - antecedent

                support_itemset = frequent_itemsets[itemset]
                support_antecedent = frequent_itemsets[antecedent]
                support_consequent = frequent_itemsets[consequent]

                confidence = support_itemset / support_antecedent
                lift = confidence / support_consequent

                if confidence >= min_confidence:
                    rules.append((antecedent, consequent,
                                  support_itemset,
                                  confidence,
                                  lift))
    return rules

In [23]:
rules = generate_rules_with_lift(frequent_itemsets, transactions, 0.1)

for ant, cons, supp, conf, lift in rules:
    print(f"{set(ant)} → {set(cons)} | "
          f"Support: {supp:.3f} | "
          f"Confidence: {conf:.3f} | "
          f"Lift: {lift:.3f}")

{'other vegetables'} → {'whole milk'} | Support: 0.015 | Confidence: 0.122 | Lift: 0.769
{'yogurt'} → {'whole milk'} | Support: 0.011 | Confidence: 0.130 | Lift: 0.823
{'rolls/buns'} → {'whole milk'} | Support: 0.014 | Confidence: 0.127 | Lift: 0.804
{'soda'} → {'whole milk'} | Support: 0.012 | Confidence: 0.120 | Lift: 0.758
